In [29]:
from aiida_quantumespresso.workflows.pw.base import PwBaseWorkChain

spec = PwBaseWorkChain.spec()

label = spec.inputs.ports['metadata'].ports['label']



In [30]:
from dataclasses import asdict, is_dataclass
from datetime import date, datetime
from enum import Enum
from pathlib import Path
from typing import Any, Mapping, Sequence

from aiida import load_profile, orm
from aiida.common.exceptions import MissingEntryPointError
from aiida.engine import submit
from aiida.engine.processes.ports import InputPort, PortNamespace
from aiida.plugins import WorkflowFactory
from aiida.plugins.entry_point import get_entry_point_names


def _type_to_string(value: Any) -> str:
    """Render a class/type declaration into a compact string."""
    if value is None:
        return "Any"

    if isinstance(value, tuple):
        return " | ".join(_type_to_string(entry) for entry in value)

    if isinstance(value, type):
        return value.__name__

    return str(value)


def _to_jsonable(value: Any) -> Any:
    """Convert arbitrary python/AiiDA values into JSON-serializable values."""
    if value is None or isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, Enum):
        return value.value

    if isinstance(value, (date, datetime)):
        return value.isoformat()

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, orm.Node):
        return {
            "pk": int(value.pk),
            "uuid": str(value.uuid),
            "type": value.__class__.__name__,
        }

    if isinstance(value, Mapping):
        return {str(key): _to_jsonable(item) for key, item in value.items()}

    if isinstance(value, (list, tuple, set, frozenset)):
        return [_to_jsonable(item) for item in value]

    if is_dataclass(value):
        return _to_jsonable(asdict(value))

    if callable(value):
        module = getattr(value, "__module__", "")
        qualname = getattr(value, "__qualname__", repr(value))
        return f"<callable {module}.{qualname}>"

    return str(value)

def _extract_default(port: InputPort | PortNamespace) -> Any:
    """Return a JSON-safe default value if defined, otherwise None."""
    has_default = False
    has_default_method = getattr(port, "has_default", None)
    if callable(has_default_method):
        try:
            has_default = bool(has_default_method())
        except Exception:  # noqa: BLE001
            has_default = False

    if not has_default:
        return None

    try:
        default_value = port.default
    except Exception:  # noqa: BLE001
        return None

    return _to_jsonable(default_value)


def _extract_help(port: InputPort | PortNamespace) -> str | None:
    try:
        text = port.help
    except Exception:  # noqa: BLE001
        return None
    return str(text) if text else None
def _extract_required(port: InputPort | PortNamespace) -> bool:
    try:
        return bool(port.required)
    except Exception:  # noqa: BLE001
        return False
    
def serialize_spec(port_or_namespace: InputPort | PortNamespace, name: str = "inputs") -> dict[str, Any]:
    """
    Recursively serialize an AiiDA input port/namespace into a JSON-safe structure.
    """
    payload: dict[str, Any] = {
        "name": name,
        "required": _extract_required(port_or_namespace),
        "default": _extract_default(port_or_namespace),
        "help": _extract_help(port_or_namespace),
    }

    if isinstance(port_or_namespace, PortNamespace):
        payload["type"] = "PortNamespace"
        payload["dynamic"] = bool(getattr(port_or_namespace, "dynamic", False))
        payload["ports"] = {
            child_name: serialize_spec(child_port, child_name)
            for child_name, child_port in port_or_namespace.items()
        }
    else:
        payload["type"] = _type_to_string(getattr(port_or_namespace, "valid_type", None))
        payload["non_db"] = bool(getattr(port_or_namespace, "non_db", False))

    return payload


In [31]:
serialize_spec(PwBaseWorkChain.spec().inputs)

{'name': 'inputs',
 'required': True,
 'default': None,
 'help': None,
 'type': 'PortNamespace',
 'dynamic': False,
 'ports': {'metadata': {'name': 'metadata',
   'required': False,
   'default': None,
   'help': None,
   'type': 'PortNamespace',
   'dynamic': False,
   'ports': {'store_provenance': {'name': 'store_provenance',
     'required': False,
     'default': True,
     'help': 'If set to `False` provenance will not be stored in the database.',
     'type': 'bool',
     'non_db': False},
    'description': {'name': 'description',
     'required': False,
     'default': None,
     'help': 'Description to set on the process node.',
     'type': 'str | NoneType',
     'non_db': False},
    'label': {'name': 'label',
     'required': False,
     'default': None,
     'help': 'Label to set on the process node.',
     'type': 'str | NoneType',
     'non_db': False},
    'call_link_label': {'name': 'call_link_label',
     'required': False,
     'default': 'CALL',
     'help': 'The la

In [51]:
builder = PwBaseWorkChain.get_builder()

builder.pw


{'metadata': {'options': {'stash': {}}}, 'monitors': {}, 'pseudos': {}}